# 🔍 Validação das Views

## 📦 Análise de Pedidos — Olist E-commerce

Análise exploratória da tabela de pedidos focando em status, 
volume e evolução temporal.

### Célula 2 — Imports e carregamento:

In [2]:
import pandas as pd

pd.set_option('display.float_format', '{:.2f}'.format)

pedidos = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

pedidos.shape


(99441, 8)

## 📊 Análise de Status dos Pedidos

In [2]:
# value_counts() — conta a frequência de cada valor único em uma coluna
# normalize=True — retorna em percentual em vez de quantidade absoluta

pedidos['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## 📊 Validando vw_status_pedidos

In [8]:
# Extraindo ano e mês da data de compra
# dt.to_period('M') — converte para período mensal (2017-01, 2017-02...)


def get_status_pedidos(pedidos):

    status_traducao = {
        'delivered':   'Entregue',
        'shipped':     'Em Transporte',
        'canceled':    'Cancelado',
        'unavailable': 'Indisponível',
        'invoiced':    'Faturado',
        'processing':  'Em Processamento',
        'created':     'Criado',
        'approved':    'Aprovado'
    }

    df = pedidos.copy()
    df['status_pt']  = df['order_status'].map(status_traducao)
    df['ano']        = df['order_purchase_timestamp'].dt.year
    df['mes']        = df['order_purchase_timestamp'].dt.month
    df['ano_mes']    = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

    resultado = (df.groupby(['ano', 'mes', 'ano_mes', 'status_pt'])
                   .size()
                   .reset_index(name='quantidade'))

    total_mes = resultado.groupby('ano_mes')['quantidade'].transform('sum')
    resultado['percentual'] = (resultado['quantidade'] / total_mes * 100).round(2)

    return resultado




In [9]:
df_status = get_status_pedidos(pedidos)
df_status.head()

,ano,mes,ano_mes,status_pt,quantidade,percentual
0,2016,9,2016-09,Cancelado,2,50.00
1,2016,9,2016-09,Em Transporte,1,25.00
2,2016,9,2016-09,Entregue,1,25.00
3,2016,10,2016-10,Cancelado,24,7.41
4,2016,10,2016-10,Em Processamento,2,0.62


### Testando a View

In [21]:
import sys
sys.path.append('..')

from views.vw_status_pedidos import get_status_pedidos
df_status = get_status_pedidos(pedidos)
df_status.head()

,ano,mes,ano_mes,status_pt,quantidade,percentual
0,2016,9,2016-09,Cancelado,2,50.00
1,2016,9,2016-09,Em Transporte,1,25.00
2,2016,9,2016-09,Entregue,1,25.00
3,2016,10,2016-10,Cancelado,24,7.41
4,2016,10,2016-10,Em Processamento,2,0.62
